In [1]:
import asyncio

import httpx

from a2a.client import ClientConfig, create_client
from a2a.client.card_resolver import A2ACardResolver
from a2a.helpers import new_text_message
from a2a.types import Role, SendMessageRequest

In [2]:
from IPython.display import display, Markdown

In [3]:
BASE_URL = "http://localhost:8001"

In [4]:
async with httpx.AsyncClient() as http:
    card = await A2ACardResolver(http, BASE_URL).get_agent_card()

In [5]:
card

name: "CyberSecurity Agent"
description: "It will answer to any cybersecurity related query"
supported_interfaces {
  url: "http://localhost:8001/"
  protocol_binding: "JSONRPC"
}
version: "0.1.0"
capabilities {
  streaming: false
  push_notifications: false
}
default_input_modes: "text/plain"
default_output_modes: "text/plain"

In [9]:
text = "Tell me about Blind SQL Injection"

In [10]:
async with httpx.AsyncClient(timeout=None) as http:
    client = await create_client(
        card,
        client_config = ClientConfig(
            supported_protocol_bindings = [
                card.supported_interfaces[0].protocol_binding
                # TransportProtocol.JSONRPC
            ],
            httpx_client = http,
        ),
    )

    try:

        request = SendMessageRequest(
            message=new_text_message(
                text=text, 
                role=Role.ROLE_USER,
            ),
        )
        async for reply in client.send_message(request):
            response = reply
            break
    finally:
        await client.close()

In [11]:
print(response)

task {
  id: "f07482ff-754e-4548-a6ec-d650aca67188"
  context_id: "4f205c7b-6541-480a-9a4b-94b2f3931144"
  status {
    state: TASK_STATE_COMPLETED
    message {
      message_id: "63943cac-edbb-47ba-915c-558ff48aa383"
      context_id: "4f205c7b-6541-480a-9a4b-94b2f3931144"
      task_id: "f07482ff-754e-4548-a6ec-d650aca67188"
      role: ROLE_AGENT
      parts {
        text: "# Blind SQL Injection: In-Depth Report\n\n## 1. Introduction\n\nBlind SQL Injection is a subtype of SQL Injection attack where an attacker can extract data from a database by sending SQL queries that cause the application to behave differently (e.g., in response times or returned content) without directly displaying the database output. Unlike classic SQL Injection (error‑based or UNION‑based), blind injection does not rely on error messages or visible data from the database; instead, it exploits subtle changes in the application’s response to infer information bit by bit.\n\n## 2. How Blind SQL Injection Works

In [12]:
display(Markdown(response.task.status.message.parts[0].text))

# Blind SQL Injection: In-Depth Report

## 1. Introduction

Blind SQL Injection is a subtype of SQL Injection attack where an attacker can extract data from a database by sending SQL queries that cause the application to behave differently (e.g., in response times or returned content) without directly displaying the database output. Unlike classic SQL Injection (error‑based or UNION‑based), blind injection does not rely on error messages or visible data from the database; instead, it exploits subtle changes in the application’s response to infer information bit by bit.

## 2. How Blind SQL Injection Works

Blind SQL Injection typically occurs when an application is vulnerable to SQL injection but the database output is not directly returned to the user. The attacker must ask the database a series of yes/no questions (or true/false conditions) and observe the application’s behavior to deduce the answer.

Two primary techniques exist:

### 2.1 Boolean-Based Blind SQL Injection
The attacker sends a SQL query with a condition that evaluates to either `TRUE` or `FALSE`. The application’s response differs between the two states (e.g., one page returns content, another returns a different page or an error). By observing these differences, the attacker can infer whether a condition is true.

**Example (in URL parameter):**
```
http://example.com/product?id=1 AND 1=1   -- True → page loads normally
http://example.com/product?id=1 AND 1=2   -- False → page may be empty or show error
```

The attacker can then progressively extract data:
- `id=1 AND (SELECT SUBSTRING(password,1,1) FROM users WHERE username='admin') = 'a'` → True/False

### 2.2 Time-Based Blind SQL Injection
If the application shows no visible difference between TRUE and FALSE conditions, the attacker introduces a time delay (e.g., using `SLEEP()` or `WAITFOR DELAY`) to create a measurable difference. If a condition is true, the server pauses; if false, it responds immediately.

**Example (MySQL):**
```
id=1 AND IF(SUBSTRING(password,1,1)='a', SLEEP(5), 0)
```
By measuring response time, the attacker can brute‑force characters.

## 3. Detailed Attack Workflow

1. **Identify the injection point** – Usually a parameter (GET/POST) that is directly concatenated into an SQL query without sanitization.
2. **Confirm blind vulnerability** – Send a TRUE condition (e.g., `1=1`) and a FALSE condition (`1=2`) and observe a consistent difference.
3. **Extract database metadata** – Determine the database version, table names, column names, etc., using Boolean logic and substring extraction.
4. **Extract data** – Brute‑force character by character. For each position, test all possible characters (e.g., 0–9, a–z, special characters) until the condition returns TRUE.
5. **Automation** – Tools like `sqlmap` automate the entire process and can extract the entire database schema and content.

### 3.1 Example of Manual Boolean Extraction
Assuming an application returns “Product found” when `id=1 AND 1=1` and “Not found” when `id=1 AND 1=2`.

To get the first character of the database name:
```
id=1 AND SUBSTRING(database(),1,1)='a'   → returns "Not found"
id=1 AND SUBSTRING(database(),1,1)='b'   → returns "Not found"
...
id=1 AND SUBSTRING(database(),1,1)='m'   → returns "Product found" (MySQL databases often start with 'm')
```

Each correct guess takes one request. For a 10‑character database name, an average of 1280 requests (10 × 128 ASCII possibilities / 2) would be needed in the worst case.

## 4. Common Database Functions Used

- **Substring extraction:** `SUBSTRING(string, start, length)` (MySQL), `SUBSTR()` (Oracle), `SUBSTRING()` (SQL Server)
- **ASCII/char conversion:** `ASCII('a')` or `CHAR(97)` to compare numeric values (reduces character set to numbers 32–126)
- **Conditional delays:**
  - MySQL: `SLEEP(seconds)`
  - SQL Server: `WAITFOR DELAY '0:0:5'`
  - PostgreSQL: `pg_sleep(seconds)`
  - Oracle: `DBMS_LOCK.SLEEP(seconds)` (requires privileges) or heavy queries (e.g., `SELECT COUNT(*) FROM all_objects`)

## 5. Detection & Prevention

### 5.1 Detection
- **Dynamic Application Security Testing (DAST)** tools send payloads and monitor response differences.
- **Static Code Analysis** to find unsanitized user input concatenated in SQL queries.
- **Log monitoring** – Look for abnormal percentages of TRUE/FALSE requests or high response times for a single parameter.

### 5.2 Prevention
1. **Use parameterized queries (prepared statements)** – Never concatenate user input directly into SQL.
2. **Input validation** – Whitelist acceptable values (e.g., integers for IDs).
3. **Least privilege principle** – Database user should have minimal permissions; restrict access to `information_schema` and system tables.
4. **Web Application Firewall (WAF)** – Can block known blind SQL injection patterns (e.g., `SLEEP`, `OR 1=1`).
5. **Disable error messages** – Prevent attackers from seeing database errors.
6. **Rate limiting** – Protect against brute‑force enumeration.

## 6. Impact

| Impact Area | Consequences |
|-------------|--------------|
| **Data breach** | Extraction of passwords, credit cards, personal data. |
| **Authentication bypass** | Login forms with blind injection can be bypassed without returning data. |
| **Database corruption** | Blind injection can be used to run `UPDATE` or `DELETE` statements (if the database user has write privileges). |
| **Privilege escalation** | Extract admin credentials or modify user roles. |
| **Denial of Service** | Time‑based injection can overload the database with delays. |

## 7. Real-World Example (CVE‑2023‑XXXX)

Many legacy applications (e.g., older PHP + MySQL apps) suffer from blind injection in search fields. For instance, an e‑commerce site with a product search like:
```php
$query = "SELECT * FROM products WHERE name LIKE '%".$_GET['search']."%'";
```
An attacker could use boolean‑based injection to enumerate product inventory or extract customer email addresses from a related database table.

## 8. Conclusion

Blind SQL Injection is a stealthy but powerful attack vector. While slower than in‑band injection, it allows an attacker to fully reconstruct a database without any visible output. Defenders must treat every user input as untrusted and enforce parameterized queries, combined with solid security practices such as least privilege, input validation, and robust monitoring.

For further reading, study the OWASP Guide on SQL Injection or practice in a controlled lab environment (e.g., TryHackMe, PortSwigger Web Security Academy).